## Transformation Guide to SysML v2 Workflows

### 1. Introduction

This notebook presents the transformation pipeline used to convert existing workflow representations into SysML v2 models. The objective is to provide a deterministic and reproducible approach for generating structured SysML v2 workflows.

The transformations are applied to multiple source datasets, including graph-based workflows **WorfBench**, BPMN models **SAP-SAM**, and SOP-based descriptions **SOP-Bench**. Each workflow is parsed to extract its core components (actions, dependencies, and control structures), which are then mapped into equivalent SysML v2 constructs such as actions, decisions, forks, and joins.

This notebook serves as a practical guide to illustrate the transformation logic, and provide examples of the resulting SysML v2 representations.

The implementation avoids the use of LLMs to ensure reproducibility and prevent semantic inconsistencies during the transformation process.

### Helper : Loading Python Modules from File Paths

In [3]:
from pathlib import Path
import importlib.util
import json
import sys

ROOT = Path.cwd()

def load_module(module_name: str, file_path: Path):
    spec = importlib.util.spec_from_file_location(module_name, str(file_path))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

### 2. Transformation of WorfBench:

**A. Load the dataset:**

In [4]:
from datasets import load_dataset

ds = load_dataset("zjunlp/WorFBench_train")

print("Splits disponibles:", ds.keys())
print("Nombre d'instances train:", len(ds["train"]))
print("Champs:", ds["train"].column_names)

C:\Users\y25elmou\PycharmProjects\ParseDatasets\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Splits disponibles: dict_keys(['train'])
Nombre d'instances train: 18679
Champs: ['messages', 'source']


**B. Extract user prompt, source, nodes and edges:**

In [5]:
worf_dir = ROOT / "WorfBench"
if str(worf_dir) not in sys.path:
    sys.path.insert(0, str(worf_dir))

worfbench = load_module("worfbench_module", worf_dir / "worfbench.py")

row_index = 0
instance = ds["train"][row_index]

user_prompt, source = worfbench.extract_user_prompt_and_source(instance)
graph = worfbench.extract_nodes_and_edges(instance)

print("source:", source)
print("user_prompt:", user_prompt)
print("nodes:")
for node_id, node_text in graph["nodes"].items():
    print(node_id, "->", node_text)

print("\nedges:")
for src, dst in graph["edges"]:
    print(src, "->", dst)

source: environment/webshop
user_prompt: WebShop [SEP] Instruction: [SEP] i am looking for gray men's casual sweatpants that are machine washable with an elastic waist
nodes:
1 -> search[gray men's casual sweatpants machine washable elastic waist]
2 -> click[Men’s sweatpants product link closest to requirements]
3 -> click[appropriate size]
4 -> click[color gray]
5 -> click[Buy Now]

edges:
START -> 1
1 -> 2
2 -> 3
3 -> 4
4 -> 5
5 -> END


**C. Transform into SysML:**

In [6]:
from Transformers.graph_to_sysml import transform_into_sysml

sysml_code = transform_into_sysml(
    graph,
    package_name="WorFBenchPackage",
    process_name="WorFBenchProcess"
)

print(sysml_code)

package WorFBenchPackage {

    action def WorFBenchProcess {


        first start;

        then action action1 {
            description = "search[gray men's casual sweatpants machine washable elastic waist]";
        }
        then action action2 {
            description = "click[Men’s sweatpants product link closest to requirements]";
        }
        then action action3 {
            description = "click[appropriate size]";
        }
        then action action4 {
            description = "click[color gray]";
        }
        then action action5 {
            description = "click[Buy Now]";
        }

        then done;
    }
}



### 3. Transformation of SAP-SAM:

**A. Load the dataset:**

In [24]:
from pathlib import Path
import glob
import json
import pandas as pd


def load_sapsam_raw_model(
    csv_dir: Path,
    json_col = "Model JSON",
    csv_name = None,
    row_index = None,
):
    if not csv_dir.exists():
        raise FileNotFoundError(f"Dossier introuvable: {csv_dir}")

    #user selected model
    if csv_name is not None and row_index is not None:
        target_csv = csv_dir / csv_name
        if not target_csv.exists():
            raise FileNotFoundError(f"CSV introuvable: {target_csv}")

        df = pd.read_csv(target_csv)
        if json_col not in df.columns:
            raise KeyError(f"Colonne absente: '{json_col}' dans {csv_name}")

        if row_index < 0 or row_index >= len(df):
            raise IndexError(f"row_index hors limites: {row_index} (0..{len(df)-1})")

        raw = df.loc[row_index, json_col]
        if pd.isna(raw) or (isinstance(raw, str) and not raw.strip()):
            raise ValueError(f"JSON vide a {csv_name}:{row_index}")

        model = json.loads(raw) if isinstance(raw, str) else raw
        if not isinstance(model, dict):
            raise ValueError(f"Le JSON n'est pas un dict a {csv_name}:{row_index}")

        return model, csv_name, row_index

    # auto-pick first valid model
    csv_files = sorted(glob.glob(str(csv_dir / "*.csv")))
    if not csv_files:
        raise FileNotFoundError(f"Aucun CSV trouve dans: {csv_dir}")

    for csv_path in csv_files:
        df = pd.read_csv(csv_path)
        if json_col not in df.columns:
            continue

        for i in range(len(df)):
            raw = df.loc[i, json_col]
            if pd.isna(raw) or (isinstance(raw, str) and not raw.strip()):
                continue

            try:
                model = json.loads(raw) if isinstance(raw, str) else raw
                if isinstance(model, dict):
                    return model, Path(csv_path).name, i
            except Exception:
                continue

    raise RuntimeError("Aucun modele JSON valide trouve dans les CSV Sap-SAM.")


csv_dir = Path(r"D:\sap_sam_2022\models")

# force a specific model
select = True
select_csv = "40000.csv"
select_raw = 0

if select:
    raw_model, picked_csv, picked_row = load_sapsam_raw_model(
        csv_dir=csv_dir,
        json_col="Model JSON",
        csv_name=select_csv,
        row_index=select_raw,
    )
else:
    raw_model, picked_csv, picked_row = load_sapsam_raw_model(
        csv_dir=csv_dir,
        json_col="Model JSON",
    )

print("CSV source:", picked_csv)
print("Row index:", picked_row)
print("Keywords:", list(raw_model.keys())[:10])
print("JSON Model:\n", raw_model)

CSV source: 40000.csv
Row index: 0
Keywords: ['resourceId', 'properties', 'stencil', 'childShapes', 'bounds', 'stencilset', 'ssextensions', 'glossaryLinkRevisions']
JSON Model:
 {'resourceId': 'canvas', 'properties': {'name': '', 'documentation': '', 'auditing': '', 'monitoring': '', 'flat': '', 'version': '', 'author': '', 'language': 'English', 'targetnamespace': 'http://www.signavio.com/bpmn20', 'expressionlanguage': 'http://www.w3.org/1999/XPath', 'typelanguage': 'http://www.w3.org/2001/XMLSchema', 'creationdate': '', 'modificationdate': '', 'itemdefinitions': '', 'signals': '', 'exporter': '', 'exporterversion': '', 'defaultprocessname': '', 'properties': '', 'properties2': '', 'datainputs': '', 'dataoutputs': '', 'inputsets': '', 'outputsets': '', 'datainputset': '', 'dataoutputset': '', 'processtype': 'None', 'isclosed': False, 'isexecutable': False, 'processid': '', 'orientation': 'horizontal', 'resources': '', 'messages': '', 'errors': '', 'interfaces': '', 'namespaces': '', '

**B. Extract BPMN structure (tasks, nodes, edges):**

In [25]:
sap_dir = ROOT / "Sap-SAM BPMN"
if str(sap_dir) not in sys.path:
    sys.path.insert(0, str(sap_dir))

bpmn_parser = load_module("bpmn_parser_module", sap_dir / "bpmn_parser.py")
bpmn = bpmn_parser.extract_bpmn(raw_model)



print("\n POOLS:")
for p in bpmn["pools"].values():
    print("-", p["name"])


print("\n\n LANES:")
for l in bpmn["lanes"].values():
    print("-", l["name"], "| pool:", bpmn["pools"][l["pool"]]["name"])

print("\n\nTASKS:")
for t in bpmn.get("tasks", {}).values():
    name = " ".join(t.get("name", "").split())
    doc = t.get("documentation", "")
    task_type = t.get("task_type", "")
    print(f"- {name} [{task_type}]")
    if doc:
        clean_doc = " ".join(str(doc).split())
        print("   doc:", clean_doc)

print("\n\nSEQUENCE EDGES:")
for e in bpmn.get("edges", {}).values():
    if e.get("type") != "SequenceFlow":
        continue
    src = e.get("src")
    tgt = e.get("tgt")
    if not src or not tgt:
        continue
    print(f"{bpmn_parser.pretty_name(bpmn, src)} -> {bpmn_parser.pretty_name(bpmn, tgt)}")



 POOLS:
- P


 LANES:
- K | pool: P
- L | pool: P
- M | pool: P
- N | pool: P


TASKS:
- I [Task]
- J [Task]
- A [StartNoneEvent]
- B [Task]
- C [Task]
- D [Task]
- F [Task]
- G [Task]
- UnnamedAction [EndNoneEvent]
- O [Task]
- V [Task]
- W [Task]


SEQUENCE EDGES:
A [StartNoneEvent] -> B [Task]
B [Task] -> UnspecifiedGateway [ParallelGateway]
UnspecifiedGateway [ParallelGateway] -> I [Task]
UnspecifiedGateway [ParallelGateway] -> C [Task]
C [Task] -> D [Task]
I [Task] -> D [Task]
E [ParallelGateway] -> J [Task]
E [ParallelGateway] -> O [Task]
D [Task] -> E [ParallelGateway]
U [Exclusive_Databased_Gateway] -> V [Task]
V [Task] -> W [Task]
W [Task] -> F [Task]
F [Task] -> G [Task]
G [Task] -> H [Exclusive_Databased_Gateway]
H [Exclusive_Databased_Gateway] -> UnnamedAction [EndNoneEvent]
J [Task] -> U [Exclusive_Databased_Gateway]
O [Task] -> U [Exclusive_Databased_Gateway]
U [Exclusive_Databased_Gateway] -> W [Task]
H [Exclusive_Databased_Gateway] -> F [Task]


**C. Transform into SysML:**

In [19]:
bpmn_into_sysml = load_module("bpmn_into_sysml_module", sap_dir / "bpmn_into_sysml.py")

sysml_code = bpmn_into_sysml.sysml_code(bpmn)

print("=" * 10)
print("GENERATED SYSML CODE")
print("=" * 10)
print(sysml_code)


GENERATED SYSML CODE
private import ScalarValues::*;
private import MOSAICO::*;

package P {

    action def P {

    part def K specializes SolutionAgent {
    }

    part def L specializes SolutionAgent {
    }

    part def M specializes SolutionAgent {
    }

    part def N specializes SolutionAgent {
    }

    k : K;
    l : L;
    m : M;
    n : N;

        action I {
            description = "I";
            agent = k;
        }

        action J {
            description = "J";
            agent = k;
        }

        action A {
            description = "A";
            agent = l;
        }

        action B {
            description = "B";
            agent = l;
        }

        action C {
            description = "C";
            agent = l;
        }

        action D {
            description = "D";
            agent = l;
        }

        action F {
            description = "F";
            agent = l;
        }

        action G {
            description = "G";
   

### 4. Transformation of SOPBench :

**A. Load the dataset:**

In [10]:
sop_main = load_module("sop_main_module", ROOT / "SOP-Bench" / "main.py")
sop_data = sop_main.load_sopbench_hf(split="train")

print("Nombre d'instances train:", len(sop_data))
print("Champs:", sop_data.column_names)

Available splits: dict_keys(['train'])
Loaded 903 samples from train
Nombre d'instances train: 903
Champs: ['domain', 'user_goal', 'user_instruction', 'user_prompt', 'initial_database', 'constraints', 'constraints_original', 'constraint_parameters', 'action_should_succeed', 'directed_action_graph', 'tools', 'system_prompt']


**B. Extract user prompt, source, nodes and edges:**

In [20]:
row_index = 100
sample = sop_data[row_index]

graph = sop_main.extract_sopbench_raw(sample)

print("Description du graphe:")
print(graph["graph_description"])

print("\nNodes:")
for n in graph["nodes"]:
    print(n["id"], ":", n["name"], "| kind =", n["kind"], "| desc =", n["description"])

print("\nEdges:")
for src, dst in graph["edges"]:
    print(src, "->", dst)

Description du graphe:
Domain: bank. User goal: set_safety_box. Instruction: You are trying to reset the contents of your safety box by setting new contents, using your username, identification, and admin password to ensure successful access and update. Prompt: Hello! I would like to reset the contents of my safety box. I'd like to set the new contents to "John's new important documents". My username is "john_doe", my identification is "padoesshnwojord", and my admin password is "addoeminhnpajoss". Can you please help me with this update?

Nodes:
0 : set_safety_box | kind = action | desc = Sets the contents of the safety box. Returns true or false for successful safety box reset.
1 : and | kind = operator | desc = None
2 : authenticate_admin_password | kind = action | desc = Verifies that the entered admin password is correct for this account. Enables more functionality. Returns true or false for admin password verification.
3 : or | kind = operator | desc = None
4 : get_account_balanc

**C. Transform into SysML:**

In [21]:
transformed = sop_main.transform_or_nodes_to_fallback(graph)

pkg_name = sop_main.sanitize_name(str(sample.get("domain") or "SOPBenchPackage"))
process_name = sop_main.sanitize_name(str(sample.get("user_goal") or "Workflow"))

sysml_code = sop_main.build_sysml_from_transformed(
    transformed,
    package_name=pkg_name,
    action_def_name=process_name,
    agent_part_name="WorkflowAgent",
    agent_instance_name="workflowAgent",
)

print(sysml_code)

package bank {

    action def set_safety_box {

        part def WorkflowAgent specializes SolutionAgent {
        }

        workflowAgent : WorkflowAgent;

        action set_safety_box {
            description = "Sets the contents of the safety box. Returns true or false for successful safety box reset.";
            agent = workflowAgent;
            out success;
        }

        action authenticate_admin_password {
            description = "Verifies that the entered admin password is correct for this account. Enables more functionality. Returns true or false for admin password verification.";
            agent = workflowAgent;
            out success;
        }

        action get_account_balance {
            description = "Retrieves the bank account balance of the user's account. Returns the float account balance or None if retrieval conditions are not met.";
            agent = workflowAgent;
            out success;
        }

        action login_user {
            descr